In [1]:
import pandas as pd
import numpy as np

In [2]:
import pytesseract

pytesseract.pytesseract.tesseract_cmd = r"C:\Program Files\Tesseract-OCR\tesseract.exe"
# change the path above if your tesseract.exe is somewhere else


In [6]:
import fitz              # PyMuPDF
from pathlib import Path
from PIL import Image    # Often helpful with pytesseract
import re

# If needed, tell pytesseract where the Tesseract binary is:
# pytesseract.pytesseract.tesseract_cmd = r"/usr/bin/tesseract"  # change as needed


def extract_text_and_images_from_pdf(pdf_path, image_output_dir):
    """
    Extracts:
      1) All text directly from the PDF pages.
      2) All images from the PDF pages into `image_output_dir`.
    
    Returns:
      - pdf_text: str, all extracted text from PDF
      - image_paths: list[Path], paths to saved image files
    """
    pdf_path = Path(pdf_path)
    image_output_dir = Path(image_output_dir)
    image_output_dir.mkdir(parents=True, exist_ok=True)

    pdf_text = []
    image_paths = []

    # Open PDF
    doc = fitz.open(pdf_path)

    for page_index in range(len(doc)):
        page = doc[page_index]

        # 1. Extract text from the page
        page_text = page.get_text("text")
        pdf_text.append(page_text)

        # 2. Extract images from the page
        image_list = page.get_images(full=True)  # full=True gives more info
        for img_index, img_info in enumerate(image_list):
            xref = img_info[0]  # image reference

            img_name = f"{pdf_path.stem}_page{page_index+1}_img{img_index+1}.png"
            img_path = image_output_dir / img_name

            try:
                pix = fitz.Pixmap(doc, xref)

                # If already grayscale or RGB, fine:
                if pix.n in (1, 3):
                    pix_to_save = pix
                else:
                    # Anything else (CMYK, alpha, masks, etc.) → convert to RGB
                    pix_to_save = fitz.Pixmap(fitz.csRGB, pix)

                pix_to_save.save(img_path)
                image_paths.append(img_path)

            except Exception as e:
                # Some images (e.g. masks) still might not be savable – just skip them
                print(f"Skipping image xref {xref} on page {page_index+1}: {e}")

            finally:
                # Free objects
                try:
                    pix_to_save = None
                except NameError:
                    pass
                try:
                    pix = None
                except NameError:
                    pass


    doc.close()

    # Join all page text into one big string
    combined_pdf_text = "\n".join(pdf_text)

    return combined_pdf_text, image_paths


def ocr_images(image_paths, lang="eng"):
    """
    Performs OCR on the list of images using pytesseract.

    Returns:
      - ocr_text: str, concatenated OCR text from all images
    """
    ocr_text_chunks = []

    for img_path in image_paths:
        # Open image (PIL Image)
        with Image.open(img_path) as img:
            text = pytesseract.image_to_string(img, lang=lang)
            ocr_text_chunks.append(text)

    combined_ocr_text = "\n".join(ocr_text_chunks)
    return combined_ocr_text


def clean_text(text: str) -> str:
    """
    Simple whitespace normalizer:
      - Normalize Windows newlines
      - Collapse 3+ blank lines into 2
      - Strip trailing spaces on each line
    Adjust as you like for 'proper spacing'.
    """
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    lines = [line.rstrip() for line in text.split("\n")]
    text = "\n".join(lines)
    # Collapse 3+ newlines into 2
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text


def main():
    # Folder containing your PDFs
    pdf_folder = Path("pdfs")       # change this to your folder path
    images_root = Path("pdf_images")
    images_root.mkdir(parents=True, exist_ok=True)

    all_text_chunks = []

    # Loop over all PDFs in the folder
    for pdf_path in sorted(pdf_folder.glob("*.pdf")):
        print(f"Processing: {pdf_path.name}")

        # Each PDF gets its own image subfolder (optional but nice)
        pdf_image_dir = images_root / pdf_path.stem

        # Step 1: Extract PDF text and images using PyMuPDF
        pdf_text, image_paths = extract_text_and_images_from_pdf(pdf_path, pdf_image_dir)

        # Step 2: Extract text from images using pytesseract
        ocr_text = ocr_images(image_paths, lang="eng")

        # Optional: spacing cleanup
        pdf_text_clean = clean_text(pdf_text)
        ocr_text_clean = clean_text(ocr_text)

        # Combine for this single PDF, with a separator so you know where files begin/end
        combined_for_pdf = (
            f"\n\n===== START OF FILE: {pdf_path.name} =====\n\n"
            + pdf_text_clean
            + "\n\n"
            + ocr_text_clean
            + "\n\n===== END OF FILE: {pdf_name} =====\n\n".format(pdf_name=pdf_path.name)
        )

        all_text_chunks.append(combined_for_pdf)

    # Combine all PDFs into one big string
    all_text = "\n\n".join(all_text_chunks)
    all_text = clean_text(all_text)

    # Write to a single text file
    output_file = "all_pdfs_text.txt"
    with open(output_file, "w", encoding="utf-8") as f:
        f.write(all_text)

    print(f"Done. Combined text written to: {output_file}")


if __name__ == "__main__":
    main()


Processing: Class 1 - 1_21.pdf
Processing: Class 10 - 2_23.pdf
Processing: Class 11 - 2_25.pdf
Processing: Class 12 - 3_2.pdf
Processing: Class 13 - 3_4.pdf
Processing: Class 14 - 3_9.pdf
Processing: Class 15 - 3_11.pdf
Processing: Class 16 - 3_23.pdf
Processing: Class 17 - 3_25.pdf
Processing: Class 18 - 3_30.pdf
Processing: Class 19 - 4_1.pdf
Processing: Class 2 - 1_26.pdf
Processing: Class 20 - 4_6.pdf
Processing: Class 21 - 4_8.pdf
Processing: Class 22 - 4_13.pdf
Processing: Class 23 - 4_15.pdf
Processing: Class 24 - 4_20.pdf
Processing: Class 25 - 4_22.pdf
Processing: Class 26 - 4_27.pdf
Processing: Class 3 - 1_28.pdf
Processing: Class 4 - 2_2.pdf
Processing: Class 5 - 2_4.pdf
Processing: Class 6 ΓÇô 2_9.pdf
Processing: Class 7 - 2_11 .pdf
Processing: Class 8 - 2_17.pdf
Processing: Class 9 ΓÇô 2_18.pdf
Processing: Property Outline (2).pdf
Processing: Property Outline Spring 2025 - Copy.pdf
Processing: Property outline.pdf
Processing: Property Review Session 1 Spring 2026 - Copy.pd

In [7]:
import re
import json
from pathlib import Path
from typing import List, Dict


def load_text(path: str) -> str:
    """Read the entire text file into a single string."""
    with open(path, "r", encoding="utf-8") as f:
        return f.read()


def split_into_sections(text: str) -> List[Dict]:
    """
    Split the raw text into logical sections based on headings.

    Heuristics:
      - Chapter headings:  "Chapter 1", "CHAPTER 2", etc.
      - Case headings:     lines like "Smith v. Jones" (rudimentary pattern).
      - Section headings:  "Section 1", "Sec. 2", etc.

    Returns a list of dicts:
      [
        {
          "section_index": int,
          "section_type": "chapter" | "case" | "section" | "other",
          "section_title": str,
          "text": str
        },
        ...
      ]
    """

    # Heading patterns
    chapter_re = re.compile(r"^\s*Chapter\s+\d+\b", re.IGNORECASE)
    section_re = re.compile(r"^\s*(Section|Sec\.)\s+[\w.-]+\b", re.IGNORECASE)
    # Very simple "X v. Y" pattern for case headings
    case_re = re.compile(r"^\s*[A-Z][A-Za-z0-9 .,&'-]+ v\. [A-Z][A-Za-z0-9 .,&'-]+")

    lines = text.splitlines()

    sections: List[Dict] = []
    current_lines: List[str] = []
    current_title: str = "Preamble"
    current_type: str = "other"
    section_index = 0

    def push_section():
        nonlocal section_index, current_lines, current_title, current_type
        body = "\n".join(current_lines).strip()
        if body:
            section_index += 1
            sections.append(
                {
                    "section_index": section_index,
                    "section_type": current_type,
                    "section_title": current_title,
                    "text": body,
                }
            )
        current_lines = []

    for line in lines:
        stripped = line.strip()

        # If line is empty, just accumulate it
        if not stripped:
            current_lines.append(line)
            continue

        # Check if this line is a heading
        if chapter_re.match(stripped):
            # Close previous section
            push_section()
            current_title = stripped
            current_type = "chapter"
            # Include the heading line at the top of the new section text (for context)
            current_lines = [stripped]
        elif case_re.match(stripped):
            push_section()
            current_title = stripped
            current_type = "case"
            current_lines = [stripped]
        elif section_re.match(stripped):
            push_section()
            current_title = stripped
            current_type = "section"
            current_lines = [stripped]
        else:
            # Regular content line
            current_lines.append(line)

    # Push the last section
    push_section()

    return sections


def chunk_text(
    text: str,
    max_tokens: int = 800,
    overlap: int = 200
) -> List[str]:
    """
    Split a long text into overlapping word-based chunks.

    - max_tokens: approx max number of words per chunk
    - overlap: how many words to overlap between consecutive chunks
    """
    words = text.split()
    n = len(words)
    chunks = []

    if n == 0:
        return chunks

    start = 0
    while start < n:
        end = min(start + max_tokens, n)
        chunk_words = words[start:end]
        chunk_text = " ".join(chunk_words).strip()
        if chunk_text:
            chunks.append(chunk_text)

        if end == n:
            break

        # Move start forward but keep some overlap for context
        start = max(0, end - overlap)

    return chunks


def build_chunks_for_file(
    path: str,
    max_tokens: int = 800,
    overlap: int = 200
) -> List[Dict]:
    """
    High-level wrapper:
    - load the file
    - split into logical sections (chapters, cases, sections, etc.)
    - chunk each section

    Returns a list of dicts:
      {
        "section_index": int,
        "section_type": str,
        "section_title": str,
        "chunk_index": int,
        "chunk_id": str,
        "text": str
      }
    """
    raw_text = load_text(path)
    sections = split_into_sections(raw_text)

    all_chunks: List[Dict] = []
    for sec in sections:
        sec_idx = sec["section_index"]
        sec_type = sec["section_type"]
        sec_title = sec["section_title"]
        sec_text = sec["text"]

        sec_chunks = chunk_text(sec_text, max_tokens=max_tokens, overlap=overlap)
        for i, chunk in enumerate(sec_chunks, start=1):
            # e.g., "case_12_chunk3" or "chapter_2_chunk1"
            type_prefix = sec_type if sec_type else "section"
            chunk_id = f"{type_prefix}_{sec_idx}_chunk{i}"
            all_chunks.append(
                {
                    "section_index": sec_idx,
                    "section_type": sec_type,
                    "section_title": sec_title,
                    "chunk_index": i,
                    "chunk_id": chunk_id,
                    "text": chunk,
                }
            )

    return all_chunks


def save_chunks_as_jsonl(chunks: List[Dict], out_path: str) -> None:
    """
    Save chunks to a JSONL file where each line is one JSON object.
    This is convenient for feeding into embedding pipelines later.
    """
    out_file = Path(out_path)
    with out_file.open("w", encoding="utf-8") as f:
        for ch in chunks:
            f.write(json.dumps(ch, ensure_ascii=False) + "\n")


if __name__ == "__main__":
    # 1. Set your input & output files here
    input_path = "all_pdfs_text.txt"   # <- your big text file
    output_path = "legal_chunks.jsonl"    # <- where to store chunks

    # 2. Tune these based on how big you want each chunk
    MAX_TOKENS = 800   # approx words per chunk
    OVERLAP = 200      # overlapping words

    # 3. Run the pipeline
    chunks = build_chunks_for_file(input_path, max_tokens=MAX_TOKENS, overlap=OVERLAP)
    print(f"Created {len(chunks)} chunks")

    # 4. Save as JSONL
    save_chunks_as_jsonl(chunks, output_path)
    print(f"Saved chunks to {output_path}")


Created 2727 chunks
Saved chunks to legal_chunks.jsonl


In [6]:
import json
from pathlib import Path
from typing import List, Dict, Tuple

import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


# Path to your JSONL chunks file from the previous script
CHUNKS_JSONL_PATH = "legal_chunks.jsonl"

# How many chunks to retrieve for a given question
TOP_K = 32  # you can change this


def load_chunks(path: str | Path) -> List[Dict]:
    """Load chunks from a JSONL file; one JSON object per line."""
    path = Path(path)
    chunks: List[Dict] = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            chunks.append(json.loads(line))
    return chunks


def build_vectorizer_and_matrix(texts: List[str]) -> Tuple[TfidfVectorizer, np.ndarray]:
    """
    Fit a TF-IDF vectorizer on the list of texts and return:
      - the fitted vectorizer
      - the document-term matrix (sparse)
    """
    vectorizer = TfidfVectorizer(
        stop_words="english",
        max_features=50000,  # cap vocab size to keep things manageable
    )
    matrix = vectorizer.fit_transform(texts)
    return vectorizer, matrix


def search_chunks(
    chunks: List[Dict],
    vectorizer: TfidfVectorizer,
    matrix,
    question: str,
    top_k: int = TOP_K,
) -> List[Dict]:
    """
    Compute similarity between the user question and every chunk.
    Returns a list of the top_k chunk dicts, sorted by similarity desc.
    """
    q_vec = vectorizer.transform([question])
    sims = cosine_similarity(q_vec, matrix)[0]  # shape (num_chunks,)

    # Get indices of top_k highest scores
    top_indices = np.argsort(sims)[::-1][:top_k]

    results: List[Dict] = []
    for idx in top_indices:
        results.append(chunks[int(idx)])
    return results


def main():
    # 1. Load chunks
    print(f"Loading chunks from {CHUNKS_JSONL_PATH} ...")
    chunks = load_chunks(CHUNKS_JSONL_PATH)
    if not chunks:
        print("No chunks found. Make sure legal_chunks.jsonl exists and is not empty.")
        return

    texts = [c["text"] for c in chunks]

    # 2. Build TF-IDF index
    print("Building TF-IDF index over chunks (this happens once per run)...")
    vectorizer, matrix = build_vectorizer_and_matrix(texts)
    print(f"Indexed {len(chunks)} chunks.")

    # 3. Ask a question
    question = input(
        "\nEnter your exam-style question (or 'quit' to exit):\n> "
    ).strip()
    if not question or question.lower() in {"q", "quit", "exit"}:
        print("No question given. Exiting.")
        return

    # 4. Retrieve top chunks
    top_chunks = search_chunks(chunks, vectorizer, matrix, question, top_k=TOP_K)
    if not top_chunks:
        print("No matching chunks found.")
        return

    # 5. Combine the raw text from chunks into a single simple text blob
    #    (no JSON, no metadata, just the original text fields)
    combined_text = "\n\n".join(ch["text"] for ch in top_chunks)

    # 6. Save to a simple text file
    out_path = Path("retrieved_chunks.txt")
    with out_path.open("w", encoding="utf-8") as f:
        f.write(combined_text)

    print(f"\nWrote text from {len(top_chunks)} chunks to: {out_path.resolve()}")
    print("You can now open 'retrieved_chunks.txt' and paste it into ChatGPT as context.")


if __name__ == "__main__":
    main()


Loading chunks from legal_chunks.jsonl ...
Building TF-IDF index over chunks (this happens once per run)...
Indexed 2727 chunks.

Wrote text from 32 chunks to: C:\Users\sageh\OneDrive\Desktop\Personal Projects\Chunky_project\retrieved_chunks.txt
You can now open 'retrieved_chunks.txt' and paste it into ChatGPT as context.
